# Machine Learning Pipeline

In [ ]:
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
import keras
from keras import layers, optimizers
import mlflow
import mlflow.keras
from mlflow.models.signature import infer_signature


## 1. Parameter Global


In [ ]:
# Parameter Global & Hyperparameter
RANDOM_STATE = 42

TICKER = "BBRI.JK"
TARGET_COL = "close"

TIME_STEPS = [10, 20, 30]
OPTIMIZERS = ["SGD", "Adam", "RMSprop"]
BATCH_SIZES = [8, 16, 32]
LEARNING_RATES = [0.01, 0.001, 0.0001]
MODELS = ["LSTM", "GRU"]

EPOCHS = 50
TRAIN_SIZE = 0.80


In [ ]:
DATA_PATH = Path(f"artifact/data/{TICKER}.parquet")
OUTPUT_DIR = Path("artifact/model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = f"Stock-Forecasting-{TICKER}"


In [ ]:
keras.utils.set_random_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)


## 2. Data Understanding


### 2.1. Memuat Data


In [ ]:
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Periode Observasi : {df['date'].min().date()} s/d {df['date'].max().date()}")
print(f"Total Baris Data  : {len(df)}")

### 2.2. Pengecekan Gap Tanggal


In [ ]:
full_date_range = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
missing_dates = full_date_range.difference(df["date"])

print(f"Jumlah Gap Tanggal : {len(missing_dates)}")

### 2.3. Visualisasi Harga Historis


In [ ]:
plt.figure(figsize=(13, 5))
sns.lineplot(data=df, x="date", y=TARGET_COL)
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.title(f"Pergerakan Harga Historis - {TICKER}")
plt.tight_layout()
plt.show()

## 3. Data Preparation


### 3.2. Pembagian Data


In [ ]:
train_len = int(len(df) * TRAIN_SIZE)
train_df = df.iloc[:train_len].copy().reset_index(drop=True)
test_df = df.iloc[train_len:].copy().reset_index(drop=True)

print(f"Jumlah Data Latih : {len(train_df)}")
print(f"Jumlah Data Uji   : {len(test_df)}")

### 3.3. Normalisasi Skala


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[[TARGET_COL]].values)
test_scaled = scaler.transform(test_df[[TARGET_COL]].values)

print(f"Min Scaler (Train)  : {scaler.data_min_[0]:.2f}")
print(f"Max Scaler (Train)  : {scaler.data_max_[0]:.2f}")

In [ ]:
def create_windowed_dataset(train_arr, test_arr, time_steps, batch_size):
    ds_train = keras.utils.timeseries_dataset_from_array(
        data=train_arr[:-1],
        targets=train_arr[time_steps:, 0],
        sequence_length=time_steps,
        batch_size=batch_size,
        shuffle=True,
        seed=RANDOM_STATE
    )

    test_inputs = np.concatenate([train_arr[-time_steps:], test_arr], axis=0)

    ds_test = keras.utils.timeseries_dataset_from_array(
        data=test_inputs[:-1],
        targets=test_inputs[time_steps:, 0],
        sequence_length=time_steps,
        batch_size=batch_size,
        shuffle=False
    )

    return ds_train, ds_test

## 4. Model & Evaluasi


### 4.1. Arsitektur Model

In [ ]:
class MLflowEpochCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None and "loss" in logs:
            mlflow.log_metric("train_loss", logs["loss"], step=epoch + 1)

def build_model(model_type, time_steps, opt_name, lr):
    keras.backend.clear_session()

    if opt_name == "SGD":
        opt = optimizers.SGD(learning_rate=lr)
    elif opt_name == "Adam":
        opt = optimizers.Adam(learning_rate=lr)
    elif opt_name == "RMSprop":
        opt = optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError(f"Optimizer {opt_name} tidak didukung.")

    model = keras.Sequential()
    model.add(layers.Input(shape=(time_steps, 1)))

    if model_type == "LSTM":
        model.add(layers.LSTM(units=50, return_sequences=False))
    elif model_type == "GRU":
        model.add(layers.GRU(units=50, return_sequences=False))
    else:
        raise ValueError(f"Tipe model {model_type} tidak valid.")

    model.add(layers.Dense(units=1))
    model.compile(optimizer=opt, loss="mean_squared_error")
    
    return model

In [ ]:
def calculate_metrics(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape
    }

### 4.2. Eksperimen Model

In [ ]:
def execute_trial(trial_params, train_scaled, test_scaled, y_actual, scaler, epochs):
    ts, opt, bs, lr, m_type = trial_params

    # 1. Siapkan windowed dataset
    ds_train, ds_test = create_windowed_dataset(
        train_arr=train_scaled,
        test_arr=test_scaled,
        time_steps=ts,
        batch_size=bs
    )

    # 2. Inisialisasi callback EarlyStopping dan MLflow Logger
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="loss",
            patience=8,
            restore_best_weights=True,
            verbose=0
        ),
        MLflowEpochCallback()
    ]

    # 3. Bangun dan latih model
    model = build_model(model_type=m_type, time_steps=ts, opt_name=opt, lr=lr)
    history = model.fit(
        ds_train,
        epochs=epochs,
        shuffle=False,
        callbacks=callbacks,
        verbose=0
    )

    actual_epochs_trained = len(history.epoch)

    # 4. Prediksi dan denormalisasi nilai target
    preds_scaled = model.predict(ds_test, verbose=0)
    preds_inv = scaler.inverse_transform(preds_scaled).flatten()

    # 5. Kalkulasi metrik evaluasi
    metrics = calculate_metrics(y_true=y_actual, y_pred=preds_inv)

    record = {
        "model": m_type,
        "time_steps": ts,
        "optimizer": opt,
        "batch_size": bs,
        "learning_rate": lr,
        "epochs_trained": actual_epochs_trained,
        **metrics
    }
    
    return record, model, ds_test

In [ ]:
param_combinations = list(itertools.product(
    TIME_STEPS,
    OPTIMIZERS,
    BATCH_SIZES,
    LEARNING_RATES,
    MODELS
))

results = []
y_actual = test_df[TARGET_COL].values
progress_bar = tqdm(param_combinations, desc=f"Grid Search (Max {EPOCHS} Epochs)", unit="trial")

# Pelacak rekor performa model terbaik per arsitektur
best_models_data = {
    "LSTM": {"rmse": float("inf"), "record": None, "model": None, "ds_test": None},
    "GRU": {"rmse": float("inf"), "record": None, "model": None, "ds_test": None}
}

# --- Eksekusi Master Parent Run di MLflow ---
with mlflow.start_run(run_name=f"{TICKER}_Grid_Search_Master") as parent_run:
    # Metadata global parent run
    mlflow.set_tags({
        "ticker": TICKER,
        "target": TARGET_COL,
        "framework": "TensorFlow/Keras",
        "pipeline_stage": "Hyperparameter Tuning",
        "total_combinations": len(param_combinations),
        "early_stopping": "patience=8"
    })

    # Pendaftaran dataset source ke MLflow
    dataset_source = mlflow.data.from_pandas(df, targets=TARGET_COL, name=f"{TICKER}_historical_data")
    mlflow.log_input(dataset_source, context="training_and_testing")

    # Iterasi pencarian kombinasi terbaik
    for params in progress_bar:
        m_ts, m_opt, m_bs, m_lr, m_name = params
        child_run_name = f"{m_name}_ts{m_ts}_{m_opt}_bs{m_bs}_lr{m_lr}"

        with mlflow.start_run(run_name=child_run_name, nested=True):
            # Log parameter individual trial
            mlflow.log_params({
                "model_type": m_name,
                "time_steps": m_ts,
                "optimizer": m_opt,
                "batch_size": m_bs,
                "learning_rate": m_lr,
                "max_epochs": EPOCHS
            })
            mlflow.set_tag("architecture", m_name)

            # Eksekusi trial dengan early stopping
            record, trained_model, ds_test = execute_trial(
                trial_params=params,
                train_scaled=train_scaled,
                test_scaled=test_scaled,
                y_actual=y_actual,
                scaler=scaler,
                epochs=EPOCHS
            )

            # Log metrik hasil dan jumlah epoch aktual yang dicapai
            mlflow.log_metric("stopped_epoch", record["epochs_trained"])
            mlflow.log_metrics({
                "RMSE": record["RMSE"],
                "MAPE": record["MAPE"],
                "MSE": record["MSE"]
            })

            results.append(record)
            current_rmse = record["RMSE"]

            # Simpan bobot terbaik per arsitektur jika rekor RMSE terpecahkan
            if current_rmse < best_models_data[m_name]["rmse"]:
                best_models_data[m_name]["rmse"] = current_rmse
                best_models_data[m_name]["record"] = record
                best_models_data[m_name]["model"] = trained_model
                best_models_data[m_name]["ds_test"] = ds_test

                save_path = OUTPUT_DIR / f"{TICKER}_{m_name}.keras"
                trained_model.save(save_path)
                mlflow.set_tag("is_current_best", "True")

            progress_bar.set_postfix({
                "Model": m_name,
                "RMSE": f"{current_rmse:.2f}",
                "Best_LSTM": f"{best_models_data['LSTM']['rmse']:.2f}",
                "Best_GRU": f"{best_models_data['GRU']['rmse']:.2f}"
            })

    # Simpan riwayat lengkap eksperimen ke file CSV dan upload sebagai artefak MLflow
    csv_output_path = OUTPUT_DIR / f"{TICKER}_hyperparameter.csv"
    results_df = pd.DataFrame(results)
    results_df.to_csv(csv_output_path, index=False)
    mlflow.log_artifact(str(csv_output_path))

### 4.3. Penentuan Paramter Terbaik

In [ ]:
grouped_by_model = results_df.groupby("model")
best_indices_series = grouped_by_model["RMSE"].idxmin()
best_indices = best_indices_series.values

best_configs_df = results_df.loc[best_indices].reset_index(drop=True)

best_configs = {}
for _, row in best_configs_df.iterrows():
    best_configs[row["model"]] = row.to_dict()

summary_best_df = best_configs_df.copy()
summary_best_df

### 4.4. Inferensi Model


In [ ]:
for model_name, cfg in best_configs.items():
    ts = int(cfg["time_steps"])
    bs = int(cfg["batch_size"])

    # Muat model terbaik dari disk
    model_path = OUTPUT_DIR / f"{TICKER}_{model_name}.keras"
    loaded_model = keras.models.load_model(model_path)

    # Siapkan windowed dataset
    _, ds_test = create_windowed_dataset(train_scaled, test_scaled, ts, bs)

    # Lakukan prediksi pada data uji
    pred_test = scaler.inverse_transform(loaded_model.predict(ds_test, verbose=0)).flatten()

    # Buat plot inferensi
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(train_df["date"].iloc[ts:], train_df[TARGET_COL].iloc[ts:], label="Train Actual")
    ax.plot(test_df["date"], test_df[TARGET_COL], label="Test Actual")
    ax.plot(test_df["date"], pred_test, label=f"{model_name} Test Prediction")

    ax.set_title(f"Hasil Inferensi {model_name} - {TICKER} (RMSE: {cfg['RMSE']:.2f}, MAPE: {cfg['MAPE']:.2f}%)")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Harga (Rp)")
    ax.legend()
    plt.tight_layout()

    # Log model terbaik beserta figur visualisasi ke MLflow Run Khusus
    best_run_name = f"Best_Model_{model_name}"

    with mlflow.start_run(run_name=best_run_name):
        mlflow.log_params(cfg)
        mlflow.log_metrics({
            "best_RMSE": cfg["RMSE"],
            "best_MAPE": cfg["MAPE"],
            "best_MSE": cfg["MSE"]
        })
        mlflow.log_figure(fig, f"plots/{TICKER}_{model_name}_inference.png")
        mlflow.keras.log_model(
            model=loaded_model,
            name=f"model_{model_name.lower()}",
            registered_model_name=f"{TICKER}_{model_name}_Production"
        )

    plt.show()